# Biohub Cell Tracking V4 - UNet + Transformer + ILP Reproduction
#
This is a Kaggle-submission wrapper for the public learned baseline from:
#
- `thibautgoldsborough/unet-baseline-inference-submission`
- public artifact dataset: `thibautgoldsborough/cellmot-baseline-artifacts`
#
The goal of this version is to reproduce the current high-score public family
in our existing private Kaggle notebook slug. It runs with GPU enabled and
internet disabled. All code, weights, and dependency wheels come from the
public artifact dataset.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd


COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
TEST_DIR = COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "cellmot_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"

# Public reference defaults. The threshold was reported as the best public sweep
# point in the reference notebook. ILP is the main jump over greedy linking.
DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))

# For debugging only; keep empty for a real submission.
SLICE = os.environ.get("BIOHUB_SLICE", "").strip()

print("Biohub Cell Tracking V4 - UNet + Transformer + ILP")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(
    "Config:",
    json.dumps(
        {
            "method": METHOD,
            "weights": WEIGHTS_RELATIVE,
            "det_threshold": DET_THRESHOLD,
            "unet_batch_size": UNET_BATCH_SIZE,
            "use_ilp": USE_ILP,
            "ilp_edge_weight": ILP_EDGE_WEIGHT,
            "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
            "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
            "ilp_division_weight": ILP_DIVISION_WEIGHT,
            "slice": SLICE,
        },
        indent=2,
        sort_keys=True,
    ),
)


## Locate bundled artifacts
#
Kaggle may mount the public dataset under either the newer
`/kaggle/input/datasets/<owner>/<slug>/<slug>` path or the older
`/kaggle/input/<slug>` path. We search for a directory containing `repo`,
`weights`, and `wheels`.


In [ ]:
def find_artifacts_root() -> Path:
    explicit = os.environ.get("BIOHUB_ARTIFACTS", "").strip()
    candidates: list[Path] = []
    if explicit:
        candidates.append(Path(explicit))

    candidates.extend(
        [
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ]
    )

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            if "cellmot-baseline-artifacts" in child.name:
                candidates.append(child)
                candidates.append(child / "cellmot-baseline-artifacts")
            if child.name == "datasets":
                candidates.append(child / "thibautgoldsborough" / "cellmot-baseline-artifacts" / "cellmot-baseline-artifacts")

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "repo").exists() and (candidate / "weights").exists() and (candidate / "wheels").exists():
            return candidate

    searched = "\n".join(str(path) for path in seen)
    raise FileNotFoundError(f"Could not find cellmot-baseline-artifacts. Searched:\n{searched}")


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Artifact entries:", sorted(path.name for path in ARTIFACTS.iterdir())[:20])


## Offline install and repo setup
#
The submitted notebook has internet disabled. Dependencies are installed from
artifact wheels only, then the bundled repository and weights are copied into a
writable working directory.


In [ ]:
install_cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-index",
    "--find-links",
    str(ARTIFACTS / "wheels"),
    "tracksdata",
    "zarr>=3.0.10",
    "pyscipopt",
]
print(" ".join(install_cmd))
subprocess.run(install_cmd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(ARTIFACTS / "repo", REPO_DIR)
shutil.copytree(ARTIFACTS / "weights", REPO_DIR / "weights", dirs_exist_ok=True)
sys.path.insert(0, str(REPO_DIR / "src"))

weights_dir = REPO_DIR / "weights" / METHOD / "split_0"
print("REPO_DIR:", REPO_DIR)
print("Weights dir:", weights_dir)
print("Weights files:", sorted(path.name for path in weights_dir.iterdir()))


## Predict test graphs
#
The baseline script writes one `.geff` graph per test video under
`predictions/<dataset>/<method>/split_0/`.


In [ ]:
def list_test_stems() -> list[str]:
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory does not exist: {TEST_DIR}")
    stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr"))
    if not stems:
        raise FileNotFoundError(f"No test .zarr files found in {TEST_DIR}")
    return stems


test_stems = list_test_stems()
print(f"Found {len(test_stems)} test videos")
print(test_stems[:10])

splits_path = REPO_DIR / "kaggle_test_splits.json"
splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}], indent=2))

predict_cmd = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir",
    str(TEST_DIR),
    "--splits",
    str(splits_path.name),
    "--split",
    "0",
    "--weights",
    WEIGHTS_RELATIVE,
    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
    "--det-threshold",
    str(DET_THRESHOLD),
    "--ilp-edge-weight",
    str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight",
    str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight",
    str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight",
    str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    predict_cmd.append("--use-ilp")
if SLICE:
    predict_cmd.extend(["--slice", SLICE])

start_time = time.time()
print(" ".join(predict_cmd))
subprocess.run(predict_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)
predict_seconds = time.time() - start_time
print(f"Prediction completed in {predict_seconds / 60:.2f} minutes")


## Build `submission.csv`
#
Flatten predicted GEFF graphs into the competition table format. One row is
either a node or an edge. We also run basic structural checks before ending the
notebook, so Kaggle only submits a complete CSV.


In [ ]:
import tracksdata as td


SUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]


def graph_from_geff(path: Path):
    graph = td.graph.IndexedRXGraph.from_geff(path)
    return graph[0] if isinstance(graph, tuple) else graph


geffs = sorted((REPO_DIR / "predictions").glob(f"*/{METHOD}/split_0/*.geff"))
print(f"Found {len(geffs)} prediction graphs")
if len(geffs) != len(test_stems):
    found = {path.stem for path in geffs}
    missing = sorted(set(test_stems) - found)
    raise RuntimeError(f"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}")

rows: list[dict[str, object]] = []
stats_rows: list[dict[str, object]] = []

for geff_path in geffs:
    dataset = geff_path.stem
    graph = graph_from_geff(geff_path)
    node_count = 0
    edge_count = 0
    division_sources: dict[int, int] = {}

    for row in graph.node_attrs().iter_rows(named=True):
        rows.append(
            {
                "dataset": dataset,
                "row_type": "node",
                "node_id": int(row["node_id"]),
                "t": int(row["t"]),
                "z": int(round(float(row["z"]))),
                "y": int(round(float(row["y"]))),
                "x": int(round(float(row["x"]))),
                "source_id": -1,
                "target_id": -1,
            }
        )
        node_count += 1

    for row in graph.edge_attrs().iter_rows(named=True):
        source_id = int(row["source_id"])
        target_id = int(row["target_id"])
        rows.append(
            {
                "dataset": dataset,
                "row_type": "edge",
                "node_id": -1,
                "t": -1,
                "z": -1,
                "y": -1,
                "x": -1,
                "source_id": source_id,
                "target_id": target_id,
            }
        )
        edge_count += 1
        division_sources[source_id] = division_sources.get(source_id, 0) + 1

    stats_rows.append(
        {
            "dataset": dataset,
            "nodes": node_count,
            "edges": edge_count,
            "division_like_sources": sum(1 for count in division_sources.values() if count >= 2),
        }
    )

submission = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS)
submission.index.name = "id"

expected_datasets = set(test_stems)
actual_datasets = set(submission["dataset"].unique())
missing_datasets = sorted(expected_datasets - actual_datasets)
if missing_datasets:
    raise AssertionError(f"Missing datasets in submission: {missing_datasets[:10]}")

nodes = submission[submission["row_type"] == "node"]
edges = submission[submission["row_type"] == "edge"]
assert len(nodes) > 0, "No node rows produced"
assert set(submission["row_type"].unique()).issubset({"node", "edge"}), "Invalid row_type"
assert not submission.isna().any().any(), "NaN values found"
assert (nodes[["node_id", "t", "z", "y", "x"]] >= 0).all().all(), "Node fields must be non-negative"
assert (nodes[["source_id", "target_id"]] == -1).all().all(), "Node edge sentinels must be -1"
if len(edges):
    assert (edges[["node_id", "t", "z", "y", "x"]] == -1).all().all(), "Edge sentinel fields must be -1"
    assert (edges[["source_id", "target_id"]] >= 0).all().all(), "Edge endpoints must be non-negative node ids"

for dataset, group in submission.groupby("dataset"):
    ds_nodes = group[group["row_type"] == "node"]
    ds_edges = group[group["row_type"] == "edge"]
    node_ids = set(ds_nodes["node_id"])
    assert ds_nodes["node_id"].is_unique, f"Duplicate node_id in {dataset}"
    assert ds_edges["source_id"].isin(node_ids).all(), f"Dangling source_id in {dataset}"
    assert ds_edges["target_id"].isin(node_ids).all(), f"Dangling target_id in {dataset}"

submission.to_csv(SUBMISSION_PATH)
stats = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)
stats["predict_minutes_total"] = predict_seconds / 60.0
stats.to_csv(RUN_STATS_PATH, index=False)

print(f"Wrote {SUBMISSION_PATH} with {len(submission):,} rows")
print(f"Node rows: {len(nodes):,} | edge rows: {len(edges):,}")
print(f"Wrote {RUN_STATS_PATH}")
display(stats.describe(include="all"))
display(submission.head())
